# EverFlavor AI – Data Acquisition Notebook
**Team:** EverGlow  
**Phase 02 – Week 4: Data Acquisition and Collection**

This notebook downloads and inspects the main data sources used by EverFlavor AI.

**Nutritional Data**
- USDA FoodData Central

**Recipe & International Cuisine Data**
- Food.com Recipes and User Interactions (Kaggle)
- Hugging Face – Recipes with Nutrition


In [ ]:
!pip install -q kaggle datasets requests pandas pyarrow

## 1. USDA FoodData Central

**Source:** https://fdc.nal.usda.gov/  

Official U.S. government nutrition database.  
We use the free API to look up accurate calorie and nutrient values for individual ingredients.  
This supports the Nutritionist Agent.

In [16]:
from google.colab import userdata
import requests

# This loads the key from Secrets (you already did this)
USDA_API_KEY = userdata.get("USDA_API_KEY")

# Now use it directly — no quotes, no pasting the key
url = "https://api.nal.usda.gov/fdc/v1/foods/search"
params = {
    "api_key": USDA_API_KEY,          # ← use the variable here
    "query": "chicken breast",
    "pageSize": 3
}

response = requests.get(url, params=params)
data = response.json()

print("Status:", response.status_code)
print("Total hits:", data.get("totalHits"))
print("First result:", data["foods"][0]["description"])

Status: 200
Total hits: 21661
First result: CHICKEN BREAST


## 2. Food.com Recipes and User Interactions (Kaggle)

**Source:** https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions  

This is our **primary recipe dataset**.  
It contains thousands of recipes with ingredients, steps, and user interactions.  
We will use it for preprocessing, exploratory analysis, and later for the Chef Agent.

In [ ]:
# Install if needed (run once)
!pip install -q kagglehub


import kagglehub
import os

# Download the dataset
path = kagglehub.dataset_download("shuyangli94/food-com-recipes-and-user-interactions")

print("Path to dataset files:", path)
print("\nFiles inside the folder:")
print(os.listdir(path))

Using Colab cache for faster access to the 'food-com-recipes-and-user-interactions' dataset.
Path to dataset files: /kaggle/input/food-com-recipes-and-user-interactions

Files inside the folder:
['RAW_interactions.csv', 'ingr_map.pkl', 'PP_recipes.csv', 'RAW_recipes.csv', 'interactions_train.csv', 'interactions_test.csv', 'PP_users.csv', 'interactions_validation.csv']


In [ ]:
import pandas as pd

# Most common file name
recipes = pd.read_csv(f"{path}/RAW_recipes.csv")

print("Shape:", recipes.shape)
recipes.head(3)

Shape: (231637, 12)


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13


In [ ]:
import os
for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/food-com-recipes-and-user-interactions/RAW_interactions.csv
/kaggle/input/food-com-recipes-and-user-interactions/ingr_map.pkl
/kaggle/input/food-com-recipes-and-user-interactions/PP_recipes.csv
/kaggle/input/food-com-recipes-and-user-interactions/RAW_recipes.csv
/kaggle/input/food-com-recipes-and-user-interactions/interactions_train.csv
/kaggle/input/food-com-recipes-and-user-interactions/interactions_test.csv
/kaggle/input/food-com-recipes-and-user-interactions/PP_users.csv
/kaggle/input/food-com-recipes-and-user-interactions/interactions_validation.csv


## 3. Hugging Face – Recipes with Nutrition

**Source:** https://huggingface.co/datasets/datahiveai/recipes-with-nutrition  

This dataset provides recipes that already include nutrition information, diet labels, health labels, and cuisine type.  
It is useful for enriching our recipe data and for testing calorie calculations.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("datahiveai/recipes-with-nutrition", split="train")
print(dataset)
print(dataset[0])   # look at the first recipe

README.md:   0%|          | 0.00/2.63k [00:00<?, ?B/s]

recipes-with-nutrition.csv: reconstructing file:   0%|          |  0.00B /  450MB            

recipes-with-nutrition.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/39447 [00:00<?, ? examples/s]

Dataset({
    features: ['recipe_name', 'source', 'url', 'servings', 'calories', 'total_weight_g', 'image_url', 'diet_labels', 'health_labels', 'cautions', 'cuisine_type', 'meal_type', 'dish_type', 'ingredient_lines', 'ingredients', 'total_nutrients', 'daily_values', 'digest'],
    num_rows: 39447
})
{'recipe_name': "Classic Cabbage Slaw with Grandmother Shinn's Dressing", 'source': 'Food Network', 'url': 'https://www.foodnetwork.com/recipes/classic-cabbage-slaw-with-grandmother-shinns-dressing-recipe-1940391', 'servings': 6.0, 'calories': 511.28325, 'total_weight_g': 1239.311258661114, 'image_url': 'https://datahive-prod-dataset-products.s3.eu-central-1.amazonaws.com/dataset/346/0066d6f63e9b4ace21cfdb6a3579251a750aa31719be9b80e032f0fdfafb21c8/21126475.jpg', 'diet_labels': '["Balanced"]', 'health_labels': '["Vegetarian","Gluten-Free","Peanut-Free","Tree-Nut-Free","Soy-Free","Fish-Free","Shellfish-Free"]', 'cautions': '[]', 'cuisine_type': '["american"]', 'meal_type': '["lunch/dinner"]'

## 4. Open Food Facts (global, crowd-sourced) (not yet added)

**Source:** https://openfoodfacts.github.io/documentation/  

Global, crowd-sourced food database and API.  
We will use it as an additional source for nutrition, ingredient, and packaged-food information.

## 5. Additional Datasets (Not used in this notebook yet)

**CulinaryDB** – Structured recipes across 22 world regions  
https://cosylab.iiitd.edu.in/culinarydb/  
This dataset will be used later to explore ingredient relationships and cuisine patterns.

**RecipeDB** – 118,000+ recipes from 74 countries  
https://cosylab.iiitd.edu.in/recipedb  
Kept as an additional option if more international recipe data is needed.

In [17]:
# Example for Food.com
recipes.head(1000).to_csv("food_com_sample.csv", index=False)

# Example for Hugging Face
dataset.select(range(1000)).to_pandas().to_csv("hf_recipes_sample.csv", index=False)